# Day 05 — Pipeline Checkpoint & Quality Checks
**Member 1** | Goal: Validate the full pipeline before modelling begins

## Objectives
- Run `quality_checks.py` and review all 7 checks
- Confirm train/test split integrity
- Verify all 15 districts are present
- Run `pipeline.py` end-to-end (optional full re-run)

In [1]:
import pandas as pd
import duckdb
import sys
sys.path.append('../src')
from config import DB_PATH

con = duckdb.connect(DB_PATH)
print('Connected ✓')

Connected ✓


## 1. Run Quality Checks

In [2]:
from quality_checks import run_checks
run_checks()

[2026-04-29 09:40:59] 
[2026-04-29 09:40:59] MEMBER 3 — QUALITY CHECKS
[2026-04-29 09:40:59] =======================================================
[2026-04-29 09:40:59] 
Check 1 — Null values in features:
[2026-04-29 09:40:59]   Total rows:    375
[2026-04-29 09:40:59]   Total nulls:   0
[2026-04-29 09:40:59]   PASSED ✓
[2026-04-29 09:40:59] 
Check 2 — Train/test split:
[2026-04-29 09:40:59]   Train rows (≤2021): 330
[2026-04-29 09:40:59]   Test  rows (>2021): 45
[2026-04-29 09:40:59]   Districts:          15
[2026-04-29 09:40:59]   PASSED ✓
[2026-04-29 09:40:59] 
Check 3 — Yield range sanity:
[2026-04-29 09:40:59]   Min yield:  1.0 tonnes
[2026-04-29 09:40:59]   Max yield:  43.2 tonnes
[2026-04-29 09:40:59]   Mean yield: 18.8 tonnes
[2026-04-29 09:40:59]   PASSED ✓
[2026-04-29 09:40:59] 
Check 4 — Year range coverage:
[2026-04-29 09:40:59]   Years: 2000 – 2024
[2026-04-29 09:40:59]   Total unique years: 25
[2026-04-29 09:40:59]   PASSED ✓
[2026-04-29 09:40:59] 
Check 5 — Feature col

## 2. Database State at Checkpoint

In [3]:
tables = con.execute('SHOW TABLES').fetchall()
print(f'Tables in DuckDB: {len(tables)}')
print()
rows = []
for (t,) in tables:
    count = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    cols  = len(con.execute(f'SELECT * FROM {t} LIMIT 1').description)
    rows.append({'table': t, 'rows': count, 'cols': cols})
    print(f'  {t:<25} {count:>6} rows  {cols:>3} cols')

df_tables = pd.DataFrame(rows)
df_tables

Tables in DuckDB: 7

  clean_cotton                 375 rows    4 cols
  clean_weather             142455 rows    9 cols
  features                     375 rows   44 cols
  features_with_risk           375 rows   55 cols
  predictions                   15 rows    9 cols
  raw_cotton                   725 rows    3 cols
  raw_weather               144225 rows    9 cols


,table,rows,cols
0,clean_cotton,375,4
1,clean_weather,142455,9
2,features,375,44
3,features_with_risk,375,55
4,predictions,15,9
5,raw_cotton,725,3
6,raw_weather,144225,9


## 3. Train / Test Split Verification

In [4]:
df_feat = con.execute('SELECT * FROM features').df()

train = df_feat[df_feat['year'] <= 2021]
test  = df_feat[df_feat['year'] >  2021]

print(f'Train rows (≤2021): {len(train)}')
print(f'Test  rows (>2021): {len(test)}')
print(f'Train years: {train.year.min()} – {train.year.max()}')
print(f'Test  years: {test.year.min()}  – {test.year.max()}')
print(f'Districts in train: {train.region.nunique()}')
print(f'Districts in test:  {test.region.nunique()}')

Train rows (≤2021): 330
Test  rows (>2021): 45
Train years: 2000 – 2021
Test  years: 2022  – 2024
Districts in train: 15
Districts in test:  15


## 4. District Coverage

In [5]:
coverage = df_feat.groupby('region').agg(
    years=('year', 'count'),
    avg_yield=('yield_tonnes', 'mean'),
    min_yield=('yield_tonnes', 'min'),
    max_yield=('yield_tonnes', 'max')
).round(2)
coverage

,years,avg_yield,min_yield,max_yield
region,,,,
Aghdam district,25,18.33,2.6,33.5
Aghjabadi district,25,20.29,8.6,35.5
Barda district,25,23.28,12.4,35.0
Beylagan district,25,20.70,10.9,35.5
Bilasuvar district,25,18.95,10.1,35.9
Goranboy district,25,18.62,7.0,34.7
Imishli district,25,17.97,8.7,31.2
Kurdamir district,25,13.46,4.3,29.8
Neftchala district,25,15.96,6.9,38.6


## 5. GDD Sanity Check

In [6]:
gdd_cols = [c for c in df_feat.columns if '_GDD' in c]
print(f'GDD columns: {gdd_cols}')
for col in gdd_cols:
    neg = (df_feat[col] < 0).sum()
    print(f'  {col}: min={df_feat[col].min():.2f}  negatives={neg}')

GDD columns: ['planting_GDD', 'growing_GDD', 'boll_forming_GDD', 'harvest_GDD']
  planting_GDD: min=0.00  negatives=0
  growing_GDD: min=713.26  negatives=0
  boll_forming_GDD: min=304.82  negatives=0
  harvest_GDD: min=107.20  negatives=0


## 6. Pipeline Full Re-run (Optional)
> ⚠️ Uncomment only if you want to re-run everything from scratch.
> This will re-fetch API data (takes ~5 minutes).

In [7]:
# from pipeline import run_full_pipeline
# run_full_pipeline()

## Summary
- All 7 quality checks passed ✓
- 15 districts, 25 years of data ✓
- Train (2000–2021) / Test (2022–2024) split clean ✓
- GDD values non-negative ✓
- **Next:** `day_06_eda.ipynb` — exploratory analysis and visualisations

In [8]:
con.close() 